# Responses API Tool Use Test (`file_search`, `web_search`, streaming with tools)

**QE Perspective:** We validate tool calling via the Responses API for built-in tools (`file_search`, `web_search`), tool parameter validation, and streaming with tool definitions. This ensures tool execution, parameter handling, and streaming contracts remain stable.

- **`file_search`**: Request with `file_search` tool definition.
- **`web_search`**: Request with `web_search` tool definition.
- **Streaming with tools**: Streamed response with tool definitions.

Config: `BASE_URL`, `INFERENCE_MODEL`. Run via pytest.



In [ ]:
# Setup
import os
from openai import OpenAI, APIError

base_url = os.environ.get("BASE_URL")
model = os.environ.get("INFERENCE_MODEL")

assert base_url, "BASE_URL must be set"
assert model, "INFERENCE_MODEL must be set"

openai_base_url = base_url.rstrip("/")
openai_base_url = (
    openai_base_url if openai_base_url.endswith("/v1") else openai_base_url + "/v1"
)
client = OpenAI(api_key="no-key-needed", base_url=openai_base_url)

## Tool Use: `file_search`

Validate `file_search` tool specification in `responses.create`. When no vector store is attached, the request is validated or returns a handled status/error.


In [ ]:
fs_handled = False
try:
    response_fs = client.responses.create(
        model=model,
        input="Search files for document information.",
        tools=[{"type": "file_search"}],
    )
    fs_handled = response_fs.status in ("completed", "failed", "requires_action")
except APIError as e:
    fs_handled = e.status_code in (400, 404, 422, 500, 501)

assert fs_handled, (
    "file_search tool request must yield a valid response status or APIError"
)

## Tool Use: `web_search`

Validate `web_search` tool specification in `responses.create`.


In [ ]:
ws_handled = False
try:
    response_ws = client.responses.create(
        model=model,
        input="What are the latest features of Red Hat OpenShift?",
        tools=[{"type": "web_search"}],
    )
    ws_handled = response_ws.status in ("completed", "failed", "requires_action")
    if response_ws.status == "completed":
        assert response_ws.output, (
            "Expected output item from completed web_search response"
        )
except APIError as e:
    ws_handled = e.status_code in (400, 404, 422, 500, 501)

assert ws_handled, (
    "web_search tool request must yield a valid response or handled APIError"
)

## Streaming with tools

Validate streaming responses (`stream=True`) when tool definitions are passed.


In [ ]:
stream_tools_handled = False
try:
    stream_tools = client.responses.create(
        model=model,
        input="Provide a brief summary of artificial intelligence.",
        tools=[{"type": "web_search"}],
        stream=True,
    )
    chunks = []
    full_text = ""
    for chunk in stream_tools:
        chunks.append(chunk)
        delta = getattr(chunk, "delta", None)
        if isinstance(delta, str):
            full_text += delta
    stream_tools_handled = len(chunks) >= 1 or full_text.strip() != ""
except APIError as e:
    stream_tools_handled = e.status_code in (400, 404, 422, 500, 501)

assert stream_tools_handled, (
    "Streaming with tools must either produce chunks or raise a handled APIError"
)